# Controllable Image Generation for Inverse Problems — Colab demo

Runs the benchmark end to end on a Colab GPU: six inverse problems, two generative models
(JiT and pMF), and every reconstruction strategy in the registry — **SDEdit**, **MPC-RHC**,
**MPC-Δt**, **PnP-Flow** and **D-Flow** — all on the *same* problem instances with the
*same* generative noise.

**Runtime.** `Runtime → Change runtime type → GPU`.

| GPU | Status |
|---|---|
| **A100** | best; the default config finishes in a few minutes |
| **L4** | comfortable; recommended for routine use |
| **T4** | works, but substantially slower — reduce `num_images` or enable fewer tasks |
| CPU | structural checks only; not usable for real runs |

The whole notebook is: clone → setup → **restart once** → smoke test → run.

> **Before quoting any PnP or D-Flow number**, read `docs/methods_pnp_dflow.md`. Both
> methods were adapted to fit this benchmark's shared-initialisation invariant, and parts
> of each (the MeanFlow denoiser, D-Flow at `t0 < 1`) are research extensions with no
> published backing.

## 0. Confirm the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv

# compute_cap >= 8.0 means BF16 is available, which JiT prefers.
# On older GPUs the code falls back to FP32 (correct, just slower) and never FP16,
# which is deliberate: FP16 destabilises pixel-space generation.

## 1. Clone the repository

Replace the URL with your fork if you have one.

In [ ]:
REPO_URL = 'https://github.com/Abdelaal495/controllable-image-generation.git'
REPO_DIR = REPO_URL.rstrip('/').split('/')[-1].replace('.git', '')
print('will clone into /content/' + REPO_DIR)

In [ ]:
!git clone $REPO_URL
%cd /content/$REPO_DIR
!pwd

## 2. Install dependencies

`setup_colab.sh` detects the accelerator, installs the JAX and PyTorch stacks in an order
that works on Colab, repairs Pillow, and clones the model repositories at pinned revisions.
It is idempotent — re-running it is cheap.

In [ ]:
!bash setup_colab.sh

## 3. ⚠️ RESTART THE RUNTIME NOW

**`Runtime → Restart session`, then continue from the next cell.**

Installing JAX replaces shared libraries that the running Python process has already
imported. Without a restart you will get confusing `jaxlib` errors. This is needed
**once**; re-running the setup afterwards is a no-op because it leaves a marker file.

## 4. After the restart

In [ ]:
# `%cd` does not survive a restart, so re-enter the repository.
REPO_DIR = 'controllable-image-generation'      # edit if you cloned a fork
%cd /content/$REPO_DIR
!pwd

## 5. Hugging Face token

ImageNet-1k is a **gated** dataset. Before the first run:

1. accept the licence at <https://huggingface.co/datasets/ILSVRC/imagenet-1k>
2. create a token at <https://huggingface.co/settings/tokens>
3. add it as a Colab secret named `HF_TOKEN` (🔑 icon in the left sidebar) and enable
   notebook access

The code reads Colab secrets, then the environment, then a `.env` file — in that order.
No token is needed if you switch to `data.source: local_folder`.

In [ ]:
from google.colab import userdata
try:
    token = userdata.get('HF_TOKEN')
    print('HF_TOKEN found:', bool(token), '| length', len(token) if token else 0)
except Exception as exc:
    print('No Colab secret named HF_TOKEN:', exc)
    print('Add one via the key icon in the sidebar, or use data.source: local_folder.')

## 6. Dry run — validate the plan without loading a model

Resolves the configuration, expands every sweep, prints the atomic jobs and the warnings,
then exits. Always worth a look before committing GPU time: it catches typos, invalid
combinations and accidentally enormous sweeps in seconds.

Two things to read in the output:

* the **cost lines** — model evaluations, objective evaluations, data-fidelity gradients
  and backprops. PnP contributes **zero** backprops through the generative model; D-Flow
  contributes one per trajectory evaluation, which is what makes it the expensive one.
* the **warnings**. One per D-Flow job says it optimises an *intermediate* flow state
  because `t0 < 1`. That is expected and correct — it is the research extension being
  announced, not an error.

In [ ]:
%run run.py --config configs/experiments.yaml --dry-run

## 7. Smoke test — run this before the full sweep

`--check` runs the per-model probes against the real checkpoints, including four that
exist specifically for the new methods:

| check | what it asserts |
|---|---|
| `pnp_initial_projection` | the initial prior projection happens once and is counted separately from the `N` steps |
| `pnp_determinism` | repeated runs are bitwise identical, and changing `gamma0` does **not** re-roll the reprojection noise |
| `dflow_gradient` | `d(loss)/dq` through the trajectory is finite and non-zero |
| `dflow_optimisation` | Adam actually reduces the objective, and the output is the trajectory of the **final** `q` |

If any of these fail, stop here — the numbers from a full run would not mean anything.

In [ ]:
%run run.py --config configs/experiments.yaml --num-images 1 --experiments denoising --check

## 8. Run

`%run` (not `!python`) gives notebook-like behaviour: progress, summary tables and
matplotlib figures appear inline.

If you hit out-of-memory on a 16 GB T4, run the two model families separately —
`--models jit` then `--models pmf` — instead of letting both share the session.

In [ ]:
%run run.py --config configs/experiments.yaml

## 9. Results

Everything lands in `outputs/<run_id>/`. Each atomic job is written the moment it finishes,
so a disconnect never costs you completed work; re-running with `--run-id <existing>`
reuses what is already done.

In [ ]:
import glob, os
run_dir = sorted(glob.glob('outputs/run_*'))[-1]
print('run directory:', run_dir)
for name in sorted(os.listdir(run_dir)):
    print('  ', name)

### Quality

One row per atomic job. Nothing is averaged across models, methods or hyperparameters —
the only averaging is over the images inside a single job.

In [ ]:
import pandas as pd
pd.set_option('display.width', 200)
df = pd.read_csv(f'{run_dir}/results.csv')
ok = df[df.status == 'ok']

cols = ['task', 'model', 'method', 't0', 'steps', 'num_mpc_steps', 'K', 'lam',
        'num_pnp_steps', 'gamma0', 'alpha', 'noise_samples', 'num_opt_steps', 'lr',
        'psnr', 'ssim', 'lpips', 'measurement_rmse', 'runtime_per_image']
ok[cols].sort_values(['task', 'model', 'method'])

### Compute and memory

`backprops_through_model` counts backward passes through *generative* evaluations —
**0** for SDEdit, RHC `K=1` and every PnP job. `data_gradient_evaluations` is separate: it
counts gradients of the data-fidelity term, which for a latent model differentiates the VAE
decoder but never the generative trajectory.

The GPU columns are a **job peak at the stated batch size** and are never divided by the
batch. `gpu_incremental_peak_gib` is what the method itself added on top of an already
resident model. Empty columns mean no CUDA device or `--no-gpu-memory`; check
`gpu_memory_source` before comparing numbers across frameworks — a Torch allocator
high-water mark and a sampled NVML process peak are different measurements.

In [ ]:
cost = ['task', 'model', 'method', 'batch_size', 'model_evaluations',
        'expected_model_evaluations', 'backprops_through_model',
        'objective_evaluations', 'data_gradient_evaluations', 'optimizer_iterations',
        'denoiser_samples', 'runtime_per_image',
        'gpu_baseline_gib', 'gpu_peak_gib', 'gpu_incremental_peak_gib']
ok[cost].sort_values(['task', 'model', 'method'])

In [ ]:
# Where did each hyperparameter come from? `repository_default_untuned` means nobody
# has tuned it for this checkpoint -- including me.
prov = ['method', 'hyperparameter_source_lam', 'hyperparameter_source_lr',
        'hyperparameter_source_gamma0', 'hyperparameter_source_alpha']
ok[prov].drop_duplicates().sort_values('method')

### Figures

* `paired_deltas.png` — each non-baseline job minus its **step-matched** SDEdit baseline
* `quality_vs_cost.png` — LPIPS against runtime
* `quality_vs_memory.png` — LPIPS against the incremental GPU peak, and the two costs
  against each other
* `configurations_<model>.png` — every configuration separately, with a dashed line for the
  degraded observation (i.e. doing nothing at all)
* `<task>_<model>_page_01.png` — the actual reconstructions
* `problem_instances.png` — ground truth / degraded / guide, per task

In [ ]:
from IPython.display import Image, display
import glob
for path in sorted(glob.glob(f'{run_dir}/figures/*.png')):
    print(path)
    display(Image(filename=path))

## 10. Where to go next

Everything below is a `configs/experiments.yaml` edit — no code changes.

**Tune PnP and D-Flow before trusting them.** `gamma0`, `alpha`, `lr` and `num_opt_steps`
ship as untuned repository defaults, and every result row says so in its provenance column.
Comparing an untuned method against a tuned one measures the tuning. One task, one image,
a few minutes:

```yaml
pnp:   {t0: 0.8, num_pnp_steps: 10, gamma0: [0.1, 1.0, 10.0], alpha: [0.5, 1.0]}
dflow: {t0: 0.8, steps: 1, num_opt_steps: 10, lr: [0.003, 0.01, 0.03]}
```

**Corruption-strength sweep.** The default sits at `t0: 0.8`, which destroys 80% of the
signal; most strategies score below the degraded input there. Finding where each one
crosses that line is the more informative experiment:

```yaml
sdedit: {t0: [0.3, 0.5, 0.6, 0.8], steps: 4, solver: heun}
pnp:    {t0: [0.3, 0.5, 0.6, 0.8], num_pnp_steps: 10}
dflow:  {t0: [0.3, 0.5, 0.6, 0.8], steps: 1, num_opt_steps: 10}
```

**PnP noise averaging.** `noise_samples: [1, 5]` — the paper averages 5 denoised
realisations; this repository defaults to 1. Runtime scales with it, peak memory does not.

**D-Flow trajectory depth.** `steps: [1, 2, 4]` costs proportionally more memory, since the
whole trajectory stays in the autograd graph and there is no gradient checkpointing. Note
that `steps` is *not* comparable across families: for JiT it is Euler/Heun stages, for pMF
it is learned finite-interval transitions.

**Regularisation sweep (MPC).** `lam: [0.5, 1, 5, 15]` — the built-in defaults come from
MPC-Flow Table E2, tuned for a different model and resolution.

Always check the size first:

In [ ]:
%run run.py --config configs/experiments.yaml --dry-run

---

**Colab tips**

* Mount Drive and set `runtime.output_root` to a Drive path to keep results across sessions.
* `--num-images 1 --experiments denoising` is a fast smoke test.
* `--models jit` restricts to PyTorch only (skips JAX entirely); `--models pmf` the reverse.
* Re-running with `--run-id <existing>` reuses finished jobs instead of recomputing them.
* `--no-gpu-memory` disables the memory instrumentation if you ever need to.

For clusters (Alliance / Compute Canada), see `docs/quickstart_cluster.md`.
For what is published versus adapted in PnP-Flow and D-Flow, see
`docs/methods_pnp_dflow.md`.